# BCP Appetence Model -- Unified Label (F1-optimized)
**Banque Centrale Populaire -- PFE**

**Label unifie:** client positif s'il a souscrit a AU MOINS un des 3 produits
(MaRetraite OU Avenir MesEnfants OU Epargne Evolution)

**Optimization target:** F1 score (per-customer yes/no decision quality).
Lift@40% is retained as a secondary reference metric.

**Pipeline:**
1. Setup
2. EDA
3. Preprocessing (with leakage exclusion)
4. Feature Selection (importance + correlation + permutation)
5. Class-weighted training (`scale_pos_weight` / `class_weight` are TUNED, not fixed)
6. Optuna hyperparameter tuning per model -- **objective = F1**
7. Threshold optimization -- chosen on a tuning split, reported on a held-out test split
8. Benchmark + Lift + SHAP
9. Full-base scoring

**Key changes vs the lift-optimized version:**
- Optuna maximizes F1 instead of Lift@40%
- `scale_pos_weight` (LGBM/XGB) and `class_weight` (RF/Ada) are search parameters
- The validation set is split: one half tunes the decision threshold, the other
  half is held out for the reported F1 (no threshold leakage)
- Benchmark ranking and ensemble weights are F1-based

**Memory-safe:** sampling utilise pour les operations lourdes

---
## 0. Setup

In [ ]:
# 0.1 Set JAVA_HOME
import os, subprocess
result = subprocess.run(['find', '/usr', '-name', 'java', '-type', 'f'], capture_output=True, text=True)
java_paths = [p for p in result.stdout.strip().split('\n') if p and 'bin/java' in p]
if java_paths:
    java_home = java_paths[0].replace('/bin/java', '')
    os.environ['JAVA_HOME'] = java_home
    os.environ['PATH'] = java_home + '/bin:' + os.environ['PATH']
    print(f'JAVA_HOME: {java_home}')
else:
    print('ERROR: Java not found')

In [ ]:
# 0.2 Install packages -- mlflow pinned to match the server version (2.12.1)
subprocess.run([
    'pip', 'install', '-q',
    'pyspark==3.5.3', 'pandas', 'numpy', 'matplotlib',
    'scikit-learn', 'lightgbm', 'xgboost', 'imbalanced-learn',
    'shap', 'mlflow==2.12.1', 'optuna'
], check=True)
print('Packages installed')

In [ ]:
# 0.3 Spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName('BCP_Appetence_Unified') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark ready')

In [ ]:
# 0.4 Global config
import pandas as pd
import seaborn as sns
import numpy as np
import gc, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Paths
GOLD_PATH = 'hdfs://namenode:9000/warehouse/gold/master_table'

# Column groups
LABELS    = ['label_maRetraite', 'label_avenirMesEnfants', 'label_epargneEvolution']
DROP_COLS = ['RADICAL', 'first_account_date', 'LIBELLE_VILLE', 'TAILLE_ENTREPRI', 'has_valid_carte']
CAT_COLS  = ['GENDER', 'MARITAL_STATUS', 'CUSTOMER_RATING', 'CODE_VILLE', 'BPR']

# Leaky features (counted target products themselves)
LEAKY_COLS = ['nb_insurance_products']

# Primary metric: Lift@40%
LIFT_K = 0.40

# Sample size for memory-heavy operations
SAMPLE_N = 200_000

# Optuna config
N_OPTUNA_TRIALS = 30  # adjust based on time budget

REPORT_DIR = '/home/jovyan/work/reports'
os.makedirs(REPORT_DIR, exist_ok=True)
print(f'Reports directory: {REPORT_DIR}')

df = spark.read.parquet(GOLD_PATH)
print(f'Rows   : {df.count():,}')
print(f'Columns: {len(df.columns)}')

---
## 1. EDA

In [ ]:
# 1.1 Label distribution -- bar chart
ld = df.agg(
    F.count('RADICAL').alias('total'),
    F.sum('label_maRetraite').alias('maRetraite'),
    F.sum('label_avenirMesEnfants').alias('avenir'),
    F.sum('label_epargneEvolution').alias('epargne'),
    F.sum(F.greatest(F.col('label_maRetraite'), F.col('label_avenirMesEnfants'),
                     F.col('label_epargneEvolution'))).alias('any_product')
).toPandas()

total = ld['total'].iloc[0]
counts = {
    'MaRetraite':       int(ld['maRetraite'].iloc[0]),
    'AvenirMesEnfants': int(ld['avenir'].iloc[0]),
    'EpargneEvolution': int(ld['epargne'].iloc[0]),
    'Label unifié':     int(ld['any_product'].iloc[0]),
}
rates = {k: v/total*100 for k, v in counts.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_lbl = ['steelblue', 'darkorange', 'green', 'crimson']
axes[0].bar(counts.keys(), counts.values(), color=colors_lbl)
axes[0].set_title('Nombre de souscripteurs par produit', fontweight='bold')
axes[0].set_ylabel('Nombre de clients')
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=10)
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(rates.keys(), rates.values(), color=colors_lbl)
axes[1].set_title('Taux de souscription par produit (%)', fontweight='bold')
axes[1].set_ylabel('Taux positif (%)')
for i, (k, v) in enumerate(rates.items()):
    axes[1].text(i, v, f'{v:.2f}%', ha='center', va='bottom', fontsize=10)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/01_label_distribution.png', dpi=150)
plt.show()
ld.to_csv(f'{REPORT_DIR}/01_label_distribution.csv', index=False)
print(f'Taux positif unifié : {rates["Label unifié"]:.2f}%')

In [ ]:
# 1.2 Missing values -- horizontal bar
n_rows = df.count()
rows = []
for c in df.columns:
    n = df.filter(F.col(c).isNull()).count()
    if n > 0:
        rows.append({'column': c, 'null_pct': round(n / n_rows * 100, 2)})

if rows:
    mdf = pd.DataFrame(rows).sort_values('null_pct', ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(mdf))))
    ax.barh(mdf['column'], mdf['null_pct'], color='steelblue')
    ax.set_xlabel('Pourcentage de valeurs manquantes (%)')
    ax.set_title('Valeurs manquantes par colonne', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    for i, v in enumerate(mdf['null_pct']):
        ax.text(v, i, f' {v}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{REPORT_DIR}/02_missing_values.png', dpi=150)
    plt.show()
    mdf.to_csv(f'{REPORT_DIR}/02_missing_values.csv', index=False)
else:
    print('No missing values detected')

In [ ]:
# 1.3 Outlier percentiles
fcols = ['avg_balance', 'max_balance', 'total_flux_cred', 'total_gab_amount',
         'total_tpe_amount', 'total_retrait_amount', 'total_virement_amount',
         'savings_ratio', 'avg_monthly_spend']
rows = []
for c in fcols:
    p = df.select(F.percentile_approx(c, [0.5, 0.75, 0.95, 0.99, 1.0]).alias('p')).collect()[0]['p']
    rows.append({'column': c, 'p50': p[0], 'p75': p[1], 'p95': p[2], 'p99': p[3], 'max': p[4],
                 'ratio': round(p[4]/p[3], 1) if p[3] and p[3] > 0 else None})
pd.DataFrame(rows).to_csv(f'{REPORT_DIR}/03_outlier_percentiles.csv', index=False)
print(pd.DataFrame(rows).to_string())

In [ ]:
# 1.4 Age distribution by unified label -- bar chart
any_label = F.greatest(F.col('label_maRetraite'), F.col('label_avenirMesEnfants'),
                      F.col('label_epargneEvolution')).alias('label_any')
df_with_any = df.withColumn('label_any', any_label)
r = df_with_any.groupBy('label_any').agg(
    F.avg('age').alias('avg_age'), F.min('age').alias('min_age'),
    F.max('age').alias('max_age'), F.count('*').alias('n')
).toPandas().sort_values('label_any')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels_str = ['Non-souscripteur', 'Souscripteur']

axes[0].bar(labels_str, r['avg_age'], color=['gray', 'steelblue'])
axes[0].set_title('Âge moyen par groupe', fontweight='bold')
axes[0].set_ylabel('Âge moyen')
for i, v in enumerate(r['avg_age']):
    axes[0].text(i, v, f'{v:.1f}', ha='center', va='bottom', fontsize=11)

axes[1].bar(labels_str, r['n'], color=['gray', 'steelblue'])
axes[1].set_title('Effectifs par groupe', fontweight='bold')
axes[1].set_ylabel('Nombre de clients')
axes[1].set_yscale('log')
for i, v in enumerate(r['n']):
    axes[1].text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/04_age_by_label.png', dpi=150)
plt.show()
r.to_csv(f'{REPORT_DIR}/04_age_by_label.csv', index=False)

In [ ]:
# 1.5 Correlations -- horizontal bar chart (sorted by absolute value)
ncols = ['age', 'avg_balance', 'max_balance', 'total_flux_cred',
         'nb_gab_transactions', 'nb_tpe_transactions', 'nb_online_transactions',
         'has_digital_product', 'has_carte', 'has_pack', 'has_vignette',
         'savings_ratio', 'digital_score', 'spending_diversity',
         'product_breadth', 'balance_trend', 'avg_monthly_spend',
         'anciennete_days', 'nb_accounts']
rows = []
for col in ncols:
    try:
        rows.append({'feature': col, 'correlation': round(df_with_any.stat.corr(col, 'label_any'), 4)})
    except:
        pass
cdf = pd.DataFrame(rows)
cdf['abs_corr'] = cdf['correlation'].abs()
cdf = cdf.sort_values('abs_corr', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(cdf))))
colors_corr = ['crimson' if c < 0 else 'steelblue' for c in cdf['correlation']]
ax.barh(cdf['feature'], cdf['correlation'], color=colors_corr)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Corrélation avec le label unifié')
ax.set_title('Corrélations entre les variables et la cible', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
for i, v in enumerate(cdf['correlation']):
    ax.text(v, i, f' {v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/05_correlations.png', dpi=150)
plt.show()
cdf.drop(columns=['abs_corr']).to_csv(f'{REPORT_DIR}/05_correlations.csv', index=False)

---
## 2. Preprocessing

In [ ]:
# 2.1 Cap outliers at p99
cap_cols = ['avg_balance', 'max_balance', 'min_balance', 'last_balance',
            'total_flux_cred', 'avg_flux_cred', 'max_flux_cred',
            'total_gab_amount', 'avg_gab_amount', 'max_gab_amount',
            'total_tpe_amount', 'avg_tpe_amount', 'max_tpe_amount',
            'total_retrait_amount', 'avg_retrait_amount',
            'total_online_amount', 'avg_online_amount',
            'total_payfac_amount', 'total_virement_amount',
            'total_depot_amount', 'total_mad_amount',
            'savings_ratio', 'avg_monthly_spend', 'balance_trend']
pcts = df.select([F.percentile_approx(c, 0.99).alias(c) for c in cap_cols]).collect()[0].asDict()

df_clean = df
rep = []
for col in cap_cols:
    v = pcts[col]
    if v and v > 0:
        df_clean = df_clean.withColumn(col, F.least(F.col(col), F.lit(v)))
        rep.append({'column': col, 'cap_p99': v})
df_clean = df_clean.withColumn('NOMBRE_ENFANT', F.least(F.col('NOMBRE_ENFANT'), F.lit(10)))
rep.append({'column': 'NOMBRE_ENFANT', 'cap_p99': 10})
pd.DataFrame(rep).to_csv(f'{REPORT_DIR}/06_outlier_caps.csv', index=False)
print(f'Capped {len(rep)} columns')

In [ ]:
# 2.2 Encode + drop
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

indexers = [StringIndexer(inputCol=c, outputCol=f'{c}_idx', handleInvalid='keep') for c in CAT_COLS]
df_enc = Pipeline(stages=indexers).fit(df_clean).transform(df_clean)
df_enc = df_enc.drop(*(CAT_COLS + DROP_COLS)).fillna(0)

pd.DataFrame(
    [{'step': 'Dropped', 'column': c} for c in DROP_COLS] +
    [{'step': 'StringIndexer', 'column': c} for c in CAT_COLS]
).to_csv(f'{REPORT_DIR}/07_preprocessing_decisions.csv', index=False)
print(f'Columns after encoding: {len(df_enc.columns)}')

In [ ]:
# 2.3 Save preprocessed to HDFS
df_enc.write.mode('overwrite').parquet('hdfs://namenode:9000/warehouse/gold/master_preprocessed')
print('Saved to HDFS')

In [ ]:
# 2.4 Load + create unified label + DROP LEAKY FEATURES + split
from sklearn.model_selection import train_test_split

pdf = pd.read_parquet('/home/jovyan/work/master_preprocessed')
print(f'Shape : {pdf.shape}')
print(f'Memory: {pdf.memory_usage(deep=True).sum() / 1024**3:.2f} GB')

# Leakage exclusion: nb_insurance_products counted the 3 target products themselves
print(f'\nDropping leaky features: {LEAKY_COLS}')
FEATURE_COLS = [c for c in pdf.columns if c not in LABELS and c not in LEAKY_COLS]
print(f'Feature columns: {len(FEATURE_COLS)}')

X = pdf[FEATURE_COLS].fillna(0)

# UNIFIED LABEL: 1 if subscribed to AT LEAST one of the 3 products
y_any = ((pdf['label_maRetraite'] == 1) |
         (pdf['label_avenirMesEnfants'] == 1) |
         (pdf['label_epargneEvolution'] == 1)).astype(int)

print(f'Taux positif unifie : {y_any.mean():.4f}')
print(f'Souscripteurs (>= 1 produit) : {int(y_any.sum()):,}')

del pdf
gc.collect()

X_train, X_val, y_train, y_val = train_test_split(
    X, y_any, test_size=0.2, stratify=y_any, random_state=42
)

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
SCALE_POS_WEIGHT = n_neg / n_pos

split_df = pd.DataFrame([
    {'split': 'train', 'n': len(X_train), 'pos_rate': round(y_train.mean(), 4)},
    {'split': 'val',   'n': len(X_val),   'pos_rate': round(y_val.mean(), 4)},
])
split_df.to_csv(f'{REPORT_DIR}/08_train_val_split.csv', index=False)
print(split_df.to_string())
print(f'\nn_pos={n_pos:,}  n_neg={n_neg:,}  scale_pos_weight={SCALE_POS_WEIGHT:.1f}')
del X
gc.collect()

---
## 3. Feature Selection

**Strategy:** importance filter -> correlation filter -> permutation importance on validation set.

Permutation importance catches features that look good in training but don't generalize -- a more rigorous test than gain importance alone.

In [ ]:
# 3.1 Quick LightGBM on sample for ranking features
import lightgbm as lgb

X_train_sample = X_train.sample(n=SAMPLE_N, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

n_pos_s = int(y_train_sample.sum())
n_neg_s = len(y_train_sample) - n_pos_s

quick_model = lgb.LGBMClassifier(
    n_estimators=100, learning_rate=0.1, num_leaves=31,
    scale_pos_weight=n_neg_s / n_pos_s,
    objective='binary', random_state=42, n_jobs=-1, verbose=-1
)
quick_model.fit(X_train_sample, y_train_sample)

imp = pd.Series(quick_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

del X_train_sample, y_train_sample
gc.collect()

idf = imp.reset_index()
idf.columns = ['feature', 'importance']
idf['status'] = idf['importance'].apply(lambda x: 'zero' if x == 0 else 'kept')
idf.to_csv(f'{REPORT_DIR}/09_feature_importances.csv', index=False)

print(f'Total features      : {len(imp)}')
print(f'Zero importance     : {(imp == 0).sum()}')
print(f'Non-zero importance : {(imp > 0).sum()}')
print('\nTop 20:')
print(imp.head(20))

# Visualize gain importance
top_imp = imp.head(30).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_imp.index, top_imp.values, color='darkorange')
ax.set_xlabel('Importance par gain (LightGBM)')
ax.set_title('Top 30 variables par importance de gain', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/09_feature_importances.png', dpi=150)
plt.show()

In [ ]:
# 3.2 Correlation filter on sample (drop one of each pair with corr > 0.95)
useful = imp[imp > 0].index.tolist()
print(f'After zero-importance drop: {len(useful)}')

X_corr_sample = X_train[useful].sample(n=SAMPLE_N, random_state=42)
corr_matrix = X_corr_sample.corr().abs()
del X_corr_sample
gc.collect()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

corr_pairs = []
drop_set = set()
for col in upper.columns:
    for other in upper[col][upper[col] > 0.95].index.tolist():
        if imp[col] >= imp[other]:
            drop_set.add(other)
            corr_pairs.append({'kept': col, 'dropped': other,
                'correlation': round(upper[col][other], 4),
                'imp_kept': int(imp[col]), 'imp_dropped': int(imp[other])})
        else:
            drop_set.add(col)
            corr_pairs.append({'kept': other, 'dropped': col,
                'correlation': round(upper[col][other], 4),
                'imp_kept': int(imp[other]), 'imp_dropped': int(imp[col])})

pd.DataFrame(corr_pairs).to_csv(f'{REPORT_DIR}/10_dropped_correlated.csv', index=False)
print(f'\nDropped {len(drop_set)} correlated features')
print(pd.DataFrame(corr_pairs).to_string())

useful_after_corr = [f for f in useful if f not in drop_set]
print(f'\nAfter correlation drop: {len(useful_after_corr)}')

In [ ]:
# 3.3 Permutation importance on validation set
# This catches features that overfit (look important in train but not in val)
from sklearn.inspection import permutation_importance

X_perm_train = X_train[useful_after_corr].sample(n=SAMPLE_N, random_state=42)
y_perm_train = y_train.loc[X_perm_train.index]

n_pos_p = int(y_perm_train.sum())
n_neg_p = len(y_perm_train) - n_pos_p

model_for_perm = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=31,
    scale_pos_weight=n_neg_p / n_pos_p,
    objective='binary', random_state=42, n_jobs=-1, verbose=-1
)
model_for_perm.fit(X_perm_train, y_perm_train)

# Permutation on validation sample
X_val_sample = X_val[useful_after_corr].sample(n=50_000, random_state=42)
y_val_sample = y_val.loc[X_val_sample.index]

print('Computing permutation importance (this takes a few minutes)...')
perm = permutation_importance(
    model_for_perm, X_val_sample, y_val_sample,
    n_repeats=3, random_state=42, n_jobs=-1, scoring='average_precision'
)

perm_df = pd.DataFrame({
    'feature': useful_after_corr,
    'perm_importance': perm.importances_mean,
    'perm_std': perm.importances_std
}).sort_values('perm_importance', ascending=False)
perm_df.to_csv(f'{REPORT_DIR}/11_permutation_importance.csv', index=False)

print('\nPermutation importance (top 30):')
print(perm_df.head(30).to_string(index=False))

# Keep features with positive permutation importance (i.e. helpful on unseen data)
final_features = perm_df[perm_df['perm_importance'] > 0]['feature'].tolist()
print(f'\nFinal features (perm_importance > 0): {len(final_features)}')

pd.DataFrame({'feature': final_features}).to_csv(f'{REPORT_DIR}/12_final_features.csv', index=False)

del X_perm_train, y_perm_train, X_val_sample, y_val_sample, model_for_perm
gc.collect()

In [ ]:
# 3.4 Apply final feature selection -- via disk to avoid OOM
import pyarrow.parquet as pq

# Write filtered training and validation sets to disk, then free everything
X_train[final_features].astype('float32').to_parquet('/tmp/X_train_fs.parquet')
X_val[final_features].astype('float32').to_parquet('/tmp/X_val_fs.parquet')
y_train.to_frame('y').to_parquet('/tmp/y_train.parquet')
y_val.to_frame('y').to_parquet('/tmp/y_val.parquet')

# Free what exists
for v in ['X_train', 'X_val', 'X', 'y_train', 'y_val', 'y_any']:
    if v in globals():
        del globals()[v]
gc.collect()
print('Wrote to disk and freed memory')

# Reload from disk
X_train_fs = pd.read_parquet('/tmp/X_train_fs.parquet')
X_val_fs   = pd.read_parquet('/tmp/X_val_fs.parquet')
y_train    = pd.read_parquet('/tmp/y_train.parquet')['y']
y_val      = pd.read_parquet('/tmp/y_val.parquet')['y']

print(f'X_train_fs: {X_train_fs.shape}  memory: {X_train_fs.memory_usage(deep=True).sum()/1024**3:.2f} GB')
print(f'X_val_fs  : {X_val_fs.shape}  memory: {X_val_fs.memory_usage(deep=True).sum()/1024**3:.2f} GB')

---
## 4. Evaluation Function

Reports both ranking metrics (Lift@40%) and classification metrics (F1 at optimal threshold) since BCP wants both ranked lists AND yes/no decisions.

In [ ]:
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    precision_recall_curve, precision_score, recall_score
)
import mlflow
import mlflow.lightgbm
import mlflow.xgboost
import mlflow.sklearn

mlflow.set_tracking_uri('http://mlflow:5000')
mlflow.set_experiment('bcp_appetence_unified')

def lift_at_k(y_true, y_proba, k=0.40):
    arr = np.array(y_true)
    idx = np.argsort(y_proba)[::-1]
    n_top = int(len(arr) * k)
    cap = arr[idx[:n_top]].sum() / arr.sum()
    lift = cap / k
    return round(float(lift), 4), round(float(cap), 4)

def find_optimal_threshold(y_true, y_proba):
    """Find the threshold that maximizes F1 on the GIVEN data."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-10)
    best_idx = f1_scores[:-1].argmax()
    return float(thresholds[best_idx]), float(f1_scores[best_idx])

def evaluate(name, y_true, y_proba, k=LIFT_K, threshold=None):
    """
    Evaluate a model's probabilities.

    If `threshold` is None the F1-optimal threshold is found on (y_true, y_proba)
    itself -- convenient, but optimistic because the threshold is fit and scored
    on the same data. For an honest number, pass a threshold that was chosen on a
    SEPARATE tuning split (see the threshold-tuning cell below).
    """
    lift, cap = lift_at_k(y_true, y_proba, k)
    if threshold is None:
        threshold, _ = find_optimal_threshold(y_true, y_proba)
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'model':         name,
        'lift_40pct':    lift,
        'capture_40pct': cap,
        'pr_auc':        round(float(average_precision_score(y_true, y_proba)), 4),
        'roc_auc':       round(float(roc_auc_score(y_true, y_proba)), 4),
        'threshold':     round(float(threshold), 4),
        'f1':            round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        'precision':     round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        'recall':        round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
    }

results = []
print('Evaluation function ready')
print(f'Primary tuning metric: F1 (at per-model optimal threshold)')
print(f'Secondary metric kept for reference: Lift@{int(LIFT_K*100)}%')

---
## 5. Optuna Hyperparameter Tuning

For each model: 30 trials with Bayesian optimization (TPE sampler). Objective: maximize Lift@40% on validation set.

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------------------------
# Honest evaluation split.
# The validation set is split in two:
#   - (X_thr, y_thr)  : used to PICK the F1-optimal decision threshold
#   - (X_test, y_test): held out, used only for the final reported F1
# Optuna still optimizes against the full val set (X_val_fs / y_val), which is
# fine -- the threshold and the final score are what must not leak.
# ---------------------------------------------------------------------------
X_thr, X_test, y_thr, y_test = train_test_split(
    X_val_fs, y_val, test_size=0.5, stratify=y_val, random_state=42
)
print(f'Threshold-tuning split: {X_thr.shape}, pos_rate={y_thr.mean():.4f}')
print(f'Held-out test split   : {X_test.shape}, pos_rate={y_test.mean():.4f}')

# Subsample train for Optuna trials (each trial trains a full model)
OPTUNA_SAMPLE_N = 300_000
X_opt_train = X_train_fs.sample(n=OPTUNA_SAMPLE_N, random_state=42)
y_opt_train = y_train.loc[X_opt_train.index]

n_pos_opt = int(y_opt_train.sum())
n_neg_opt = len(y_opt_train) - n_pos_opt
spw_opt = n_neg_opt / n_pos_opt

print(f'\nOptuna train sample: {X_opt_train.shape}, pos_rate={y_opt_train.mean():.4f}')
print(f'Full-ratio scale_pos_weight (upper bound for search): {spw_opt:.1f}')
print(f'Trials per model: {N_OPTUNA_TRIALS}')

In [ ]:
# 5.1 Optuna for LightGBM -- objective = F1, scale_pos_weight tunable
def objective_lgbm(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbose': -1,
        'n_jobs': -1,
        'random_state': 42,
        # Tunable imbalance handling: search from 1.0 (balanced) up to the full
        # negative/positive ratio. Log scale samples small weights densely, where
        # the F1 optimum usually lives.
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, spw_opt, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_opt_train, y_opt_train)
    yp = model.predict_proba(X_val_fs)[:, 1]
    _, f1_opt = find_optimal_threshold(y_val, yp)
    return f1_opt

study_lgbm = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
print('Optimizing LightGBM (objective: F1)...')
study_lgbm.optimize(objective_lgbm, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'\nBest F1: {study_lgbm.best_value:.4f}')
print(f'Best params: {study_lgbm.best_params}')

pd.DataFrame(study_lgbm.trials_dataframe()).to_csv(f'{REPORT_DIR}/13a_optuna_lgbm.csv', index=False)

In [ ]:
# 5.2 Optuna for XGBoost -- objective = F1, scale_pos_weight tunable
def objective_xgb(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'verbosity': 0,
        'n_jobs': -1,
        'random_state': 42,
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, spw_opt, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-3, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_opt_train, y_opt_train, verbose=False)
    yp = model.predict_proba(X_val_fs)[:, 1]
    _, f1_opt = find_optimal_threshold(y_val, yp)
    return f1_opt

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
print('Optimizing XGBoost (objective: F1)...')
study_xgb.optimize(objective_xgb, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'\nBest F1: {study_xgb.best_value:.4f}')
print(f'Best params: {study_xgb.best_params}')

pd.DataFrame(study_xgb.trials_dataframe()).to_csv(f'{REPORT_DIR}/13b_optuna_xgb.csv', index=False)

In [ ]:
# 5.3 Optuna for Random Forest -- objective = F1, class_weight tunable
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 200),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
        # 'balanced' == full ratio (recall-heavy). Let Optuna also try milder
        # weightings and no weighting, which often win on F1.
        'class_weight': trial.suggest_categorical(
            'class_weight', ['balanced', 'balanced_subsample', None]
        ),
        'n_jobs': -1,
        'random_state': 42,
    }
    model = RandomForestClassifier(**params)
    model.fit(X_opt_train, y_opt_train)
    yp = model.predict_proba(X_val_fs)[:, 1]
    _, f1_opt = find_optimal_threshold(y_val, yp)
    return f1_opt

study_rf = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
print('Optimizing Random Forest (objective: F1)...')
study_rf.optimize(objective_rf, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'\nBest F1: {study_rf.best_value:.4f}')
print(f'Best params: {study_rf.best_params}')

pd.DataFrame(study_rf.trials_dataframe()).to_csv(f'{REPORT_DIR}/13c_optuna_rf.csv', index=False)

In [ ]:
def objective_ada(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),  # cap lower
        'learning_rate': trial.suggest_float('learning_rate', 0.3, 1.0, log=True),  # higher LR
        'random_state': 42,
    }
    max_depth = trial.suggest_int('max_depth', 2, 3)  # shallow only
    cw = trial.suggest_categorical('class_weight', ['balanced', None])
    base = DecisionTreeClassifier(max_depth=max_depth, class_weight=cw)
    model = AdaBoostClassifier(estimator=base, **params)
    try:
        model.fit(X_opt_train, y_opt_train)
    except ValueError:
        return 0.0
    yp = model.predict_proba(X_val_fs)[:, 1]
    _, f1_opt = find_optimal_threshold(y_val, yp)
    return f1_opt

# Also reduce trials and use smaller sample
N_ADA_TRIALS = 15
X_ada_train = X_opt_train.sample(n=150_000, random_state=42)
y_ada_train = y_opt_train.loc[X_ada_train.index]

# Replace X_opt_train/y_opt_train references in objective with the smaller sample
def objective_ada(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'learning_rate': trial.suggest_float('learning_rate', 0.3, 1.0, log=True),
        'random_state': 42,
    }
    max_depth = trial.suggest_int('max_depth', 2, 3)
    cw = trial.suggest_categorical('class_weight', ['balanced', None])
    base = DecisionTreeClassifier(max_depth=max_depth, class_weight=cw)
    model = AdaBoostClassifier(estimator=base, **params)
    try:
        model.fit(X_ada_train, y_ada_train)
    except ValueError:
        return 0.0
    yp = model.predict_proba(X_val_fs)[:, 1]
    _, f1_opt = find_optimal_threshold(y_val, yp)
    return f1_opt

study_ada = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
print('Optimizing AdaBoost (objective: F1)...')
study_ada.optimize(objective_ada, n_trials=N_ADA_TRIALS, show_progress_bar=True,
                   catch=(ValueError,))

---
## 6. Train Final Models with Best Params

Train on the full training set (not the Optuna subsample) using each model's best hyperparameters.

In [ ]:
# Use 500k sample from full train for final fit (manageable + representative)
FINAL_SAMPLE_N = 500_000
X_fit = X_train_fs.sample(n=FINAL_SAMPLE_N, random_state=42)
y_fit = y_train.loc[X_fit.index]

n_pos_fit = int(y_fit.sum())
n_neg_fit = len(y_fit) - n_pos_fit
spw_fit = n_neg_fit / n_pos_fit
print(f'Final training sample: {X_fit.shape}, pos_rate={y_fit.mean():.4f}')
print(f'scale_pos_weight: {spw_fit:.1f}')

In [ ]:
# 6.1 Final LightGBM
# NB: scale_pos_weight is intentionally NOT overwritten here -- the value Optuna
# tuned for F1 is already inside study_lgbm.best_params and must survive.
best_params_lgbm = study_lgbm.best_params.copy()
best_params_lgbm.update({
    'objective': 'binary', 'verbose': -1, 'n_jobs': -1,
    'random_state': 42,
})

with mlflow.start_run(run_name='LightGBM_final'):
    lgbm = lgb.LGBMClassifier(**best_params_lgbm)
    lgbm.fit(X_fit, y_fit)
    # Choose threshold on the tuning split, then report on the held-out test split
    thr_lgbm, _ = find_optimal_threshold(y_thr, lgbm.predict_proba(X_thr)[:, 1])
    y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]
    m = evaluate('LightGBM', y_test, y_proba_lgbm, threshold=thr_lgbm)
    mlflow.log_params(study_lgbm.best_params)
    mlflow.log_metrics({k: v for k, v in m.items() if isinstance(v, (int, float))})
    results.append(m)
    print(m)

In [ ]:
# 6.2 Final XGBoost
# scale_pos_weight from Optuna is preserved (not overwritten).
best_params_xgb = study_xgb.best_params.copy()
best_params_xgb.update({
    'objective': 'binary:logistic', 'eval_metric': 'aucpr',
    'verbosity': 0, 'n_jobs': -1, 'random_state': 42,
})

with mlflow.start_run(run_name='XGBoost_final'):
    xgbm = xgb.XGBClassifier(**best_params_xgb)
    xgbm.fit(X_fit, y_fit, verbose=False)
    thr_xgb, _ = find_optimal_threshold(y_thr, xgbm.predict_proba(X_thr)[:, 1])
    y_proba_xgb = xgbm.predict_proba(X_test)[:, 1]
    m = evaluate('XGBoost', y_test, y_proba_xgb, threshold=thr_xgb)
    mlflow.log_params(study_xgb.best_params)
    mlflow.log_metrics({k: v for k, v in m.items() if isinstance(v, (int, float))})
    results.append(m)
    print(m)

In [ ]:
# 6.3 Final Random Forest
# class_weight from Optuna is preserved -- no longer hardcoded to 'balanced'.
best_params_rf = study_rf.best_params.copy()
best_params_rf.update({'n_jobs': -1, 'random_state': 42})

with mlflow.start_run(run_name='RandomForest_final'):
    rf = RandomForestClassifier(**best_params_rf)
    rf.fit(X_fit, y_fit)
    thr_rf, _ = find_optimal_threshold(y_thr, rf.predict_proba(X_thr)[:, 1])
    y_proba_rf = rf.predict_proba(X_test)[:, 1]
    m = evaluate('RandomForest', y_test, y_proba_rf, threshold=thr_rf)
    mlflow.log_params(study_rf.best_params)
    mlflow.log_metrics({k: v for k, v in m.items() if isinstance(v, (int, float))})
    results.append(m)
    print(m)

In [ ]:
# 6.4 Final AdaBoost -- with fallback if best params fail on larger sample
best_params_ada = study_ada.best_params.copy()
max_depth_ada = best_params_ada.pop('max_depth')
cw_ada = best_params_ada.pop('class_weight')   # preserve tuned class_weight
best_params_ada['random_state'] = 42

def try_fit_ada(max_depth):
    base = DecisionTreeClassifier(max_depth=max_depth, class_weight=cw_ada)
    model = AdaBoostClassifier(estimator=base, **best_params_ada)
    model.fit(X_fit, y_fit)
    return model, max_depth

# Try the best max_depth, then progressively deeper trees if it fails
ada = None
used_depth = None
for d in [max_depth_ada, max_depth_ada + 1, max_depth_ada + 2, 5, 6]:
    try:
        print(f'Trying AdaBoost with max_depth={d} (class_weight={cw_ada})...')
        ada, used_depth = try_fit_ada(d)
        print(f'  Success with max_depth={d}')
        break
    except ValueError as e:
        print(f'  Failed: {e}')

if ada is None:
    raise RuntimeError('AdaBoost could not fit with any tested depth')

with mlflow.start_run(run_name='AdaBoost_final'):
    thr_ada, _ = find_optimal_threshold(y_thr, ada.predict_proba(X_thr)[:, 1])
    y_proba_ada = ada.predict_proba(X_test)[:, 1]
    m = evaluate('AdaBoost', y_test, y_proba_ada, threshold=thr_ada)
    mlflow.log_params(study_ada.best_params)
    mlflow.log_params({'max_depth_used': used_depth})
    mlflow.log_metrics({k: v for k, v in m.items() if isinstance(v, (int, float))})
    results.append(m)
    print(m)

del X_fit, y_fit
gc.collect()

---
## 7. Benchmark

In [ ]:
rdf = pd.DataFrame(results).sort_values('f1', ascending=False)
rdf.index = range(1, len(rdf) + 1)
rdf.to_csv(f'{REPORT_DIR}/18_benchmark.csv', index=False)
print('Benchmark (sorted by F1):')
print(rdf.to_string())

best_model_name = rdf.iloc[0]['model']
print(f'\nBest model (by F1): {best_model_name}')

---
## 8. Lift & Centile Analysis

In [ ]:
# NB: y_proba_* now hold predictions on the held-out TEST split (X_test),
# so all lift / centile analysis below is evaluated against y_test.
model_probas = {
    'LightGBM':     y_proba_lgbm,
    'XGBoost':      y_proba_xgb,
    'RandomForest': y_proba_rf,
    'AdaBoost':     y_proba_ada,
}

def lift_table(y_true, y_proba, label=''):
    dfl = pd.DataFrame({'y_true': np.array(y_true), 'y_proba': y_proba})
    dfl = dfl.sort_values('y_proba', ascending=False).reset_index(drop=True)
    dfl['decile'] = pd.qcut(dfl.index, 10, labels=False) + 1
    rate = np.array(y_true).mean()
    t = dfl.groupby('decile').agg(n=('y_true', 'count'), n_pos=('y_true', 'sum'),
                                   avg_score=('y_proba', 'mean')).reset_index()
    t['pos_rate'] = (t['n_pos'] / t['n']).round(4)
    t['lift'] = (t['pos_rate'] / rate).round(4)
    t['cum_cap_pct'] = (t['n_pos'].cumsum() / t['n_pos'].sum() * 100).round(2)
    t['model'] = label
    return t

def centile_analysis(y_true, y_proba, label=''):
    arr = np.array(y_true)
    idx = np.argsort(y_proba)[::-1]
    total_pos = arr.sum(); rate = arr.mean(); n = len(arr)
    rows = []
    for c in [1, 2, 5, 10, 20, 30, 40, 50]:
        nt = int(n * c / 100)
        np_c = arr[idx[:nt]].sum()
        rows.append({
            'model': label, 'centile': c, 'n': nt, 'n_pos': int(np_c),
            'capture_pct': round(float(np_c/total_pos*100), 2),
            'precision': round(float(np_c/nt if nt > 0 else 0), 4),
            'lift': round(float((np_c/nt)/rate if rate > 0 and nt > 0 else 0), 4)
        })
    return pd.DataFrame(rows)

all_lt = [lift_table(y_test, p, label=n) for n, p in model_probas.items()]
all_ct = [centile_analysis(y_test, p, label=n) for n, p in model_probas.items()]

pd.concat(all_lt).to_csv(f'{REPORT_DIR}/19_lift_tables.csv', index=False)
pd.concat(all_ct).to_csv(f'{REPORT_DIR}/20_centile_analysis.csv', index=False)

print('Centile analysis (best model):')
print(centile_analysis(y_test, model_probas[best_model_name], label=best_model_name).to_string(index=False))

In [ ]:
# Lift curves (evaluated on held-out test split)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['steelblue', 'darkorange', 'green', 'red']
for (name, proba), color in zip(model_probas.items(), colors):
    arr = np.array(y_test)
    idx = np.argsort(proba)[::-1]
    total_pos = arr.sum(); n = len(arr)
    step = max(1, n // 200)
    pp, pc, lv = [], [], []
    for i in range(1, n + 1, step):
        p = i/n*100
        c = arr[idx[:i]].sum()/total_pos*100
        pp.append(p); pc.append(c); lv.append(c/p if p > 0 else 1)
    axes[0].plot(pp, pc, color=color, lw=2, label=name)
    axes[1].plot(pp, lv, color=color, lw=2, label=name)
axes[0].plot([0, 100], [0, 100], '--', color='gray', label='Random')
axes[1].axhline(y=1, color='gray', linestyle='--', label='No lift')
for ax in axes:
    ax.axvline(x=40, color='black', linestyle=':', alpha=0.7, label='40% cutoff')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
axes[0].set(xlabel='% Population', ylabel='% Subscribers', title='Cumulative Gains')
axes[1].set(xlabel='% Population', ylabel='Lift', title='Lift Curve')
plt.suptitle('Lift Analysis -- Unified Label (test split)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/21_lift_curves.png', dpi=150)
plt.show()

---
## 9. Threshold Optimization & Yes/No Decision

For the best model, characterize the precision/recall tradeoff at different thresholds so BCP can pick one based on campaign budget.

In [ ]:
from sklearn.metrics import confusion_matrix

best_proba = model_probas[best_model_name]

# Recover the best model's probabilities on the TUNING split so the threshold
# is chosen without touching the test set.
_thr_proba_lookup = {
    'LightGBM':     lambda: lgbm.predict_proba(X_thr)[:, 1],
    'XGBoost':      lambda: xgbm.predict_proba(X_thr)[:, 1],
    'RandomForest': lambda: rf.predict_proba(X_thr)[:, 1],
    'AdaBoost':     lambda: ada.predict_proba(X_thr)[:, 1],
}
best_proba_thr = _thr_proba_lookup[best_model_name]()
opt_thr, _ = find_optimal_threshold(y_thr, best_proba_thr)

# F1 at that threshold, measured on the held-out test split (the honest number)
_yp_opt = (best_proba >= opt_thr).astype(int)
opt_f1 = f1_score(y_test, _yp_opt, zero_division=0)

# Sweep fixed thresholds on the test split for context
thresholds_to_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
thr_rows = []
for thr in sorted(thresholds_to_test + [opt_thr]):
    yp = (best_proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, yp).ravel()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    thr_rows.append({
        'threshold': round(thr, 4),
        'n_predicted_pos': int(yp.sum()),
        'pct_predicted_pos': round(yp.mean()*100, 2),
        'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        'precision': round(prec, 4),
        'recall': round(rec, 4),
        'f1': round(f1, 4),
        'is_optimal_f1': abs(thr - opt_thr) < 1e-6
    })

thr_df = pd.DataFrame(thr_rows)
thr_df.to_csv(f'{REPORT_DIR}/22_threshold_analysis.csv', index=False)
print(f'Best model: {best_model_name}')
print(f'Optimal F1 threshold (chosen on tuning split): {opt_thr:.4f}')
print(f'F1 at that threshold (held-out test split)    : {opt_f1:.4f}')
print('\nThreshold analysis (test split):')
print(thr_df.to_string(index=False))

In [ ]:
# Precision/recall curve (held-out test split)
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, best_proba)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(recall, precision, lw=2, color='steelblue')
axes[0].axhline(y=y_test.mean(), color='gray', linestyle='--', label=f'Baseline ({y_test.mean():.3f})')
axes[0].set(xlabel='Recall', ylabel='Precision', title=f'Precision-Recall Curve -- {best_model_name}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 vs threshold
f1_scores = 2 * precision * recall / (precision + recall + 1e-10)
axes[1].plot(thresholds, f1_scores[:-1], lw=2, color='darkorange')
axes[1].axvline(x=opt_thr, color='red', linestyle='--', label=f'Optimal threshold ({opt_thr:.3f})')
axes[1].set(xlabel='Threshold', ylabel='F1 score', title='F1 vs Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/23_precision_recall.png', dpi=150)
plt.show()

---
## 10. Ensemble

In [ ]:
# Weighted ensemble (weight by F1)
# Build the ensemble on BOTH splits: tune its threshold on the tuning split,
# report on the test split. We need each base model's probabilities on X_thr.
f1_scores_models = {r['model']: r['f1'] for r in results if r['model'] in model_probas}
total_f1 = sum(f1_scores_models.values())
weights = {k: v/total_f1 for k, v in f1_scores_models.items()}
print('Weights (by F1):', {k: round(v, 3) for k, v in weights.items()})

# Base-model probabilities on the tuning split
proba_thr = {
    'LightGBM':     lgbm.predict_proba(X_thr)[:, 1],
    'XGBoost':      xgbm.predict_proba(X_thr)[:, 1],
    'RandomForest': rf.predict_proba(X_thr)[:, 1],
    'AdaBoost':     ada.predict_proba(X_thr)[:, 1],
}
ens_thr = sum(weights[k] * proba_thr[k] for k in weights)
ens_test = sum(weights[k] * model_probas[k] for k in weights)

# Threshold chosen on tuning split, F1 reported on test split
ens_threshold, _ = find_optimal_threshold(y_thr, ens_thr)

with mlflow.start_run(run_name='Ensemble_Weighted'):
    m = evaluate('Ensemble_Weighted', y_test, ens_test, threshold=ens_threshold)
    mlflow.log_metrics({k: v for k, v in m.items() if isinstance(v, (int, float))})
    results.append(m)
    print(m)

fdf = pd.DataFrame(results).sort_values('f1', ascending=False)
fdf.index = range(1, len(fdf) + 1)
fdf.to_csv(f'{REPORT_DIR}/24_final_benchmark.csv', index=False)
print('\nFINAL BENCHMARK (sorted by F1):')
print(fdf.to_string())

---
## 11. Full Base Scoring (3.2M clients)

In [ ]:
# Chunked scoring to avoid OOM
CHUNK_SIZE = 500_000
pdf_full = pd.read_parquet('/home/jovyan/work/master_preprocessed')
print(f'Full dataset: {len(pdf_full):,} clients')

n = len(pdf_full)
score_lgbm = np.zeros(n)
score_xgb  = np.zeros(n)
score_rf   = np.zeros(n)
score_ada  = np.zeros(n)

for start in range(0, n, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, n)
    print(f'  chunk {start:,} -> {end:,}')
    chunk = pdf_full.iloc[start:end]
    score_lgbm[start:end] = lgbm.predict_proba(chunk[final_features].fillna(0))[:, 1]
    score_xgb[start:end]  = xgbm.predict_proba(chunk[final_features].fillna(0))[:, 1]
    score_rf[start:end]   = rf.predict_proba(chunk[final_features].fillna(0))[:, 1]
    score_ada[start:end]  = ada.predict_proba(chunk[final_features].fillna(0))[:, 1]

pdf_full['score_lgbm'] = score_lgbm
pdf_full['score_xgb']  = score_xgb
pdf_full['score_rf']   = score_rf
pdf_full['score_ada']  = score_ada
pdf_full['score_ensemble'] = (
    weights['LightGBM']     * score_lgbm +
    weights['XGBoost']      * score_xgb  +
    weights['RandomForest'] * score_rf   +
    weights['AdaBoost']     * score_ada
)
pdf_full['rank']   = pdf_full['score_ensemble'].rank(ascending=False).astype(int)
pdf_full['decile'] = pd.qcut(pdf_full['score_ensemble'], q=10, labels=False) + 1

# Yes/No prediction using the ensemble threshold tuned for F1 (ens_threshold)
pdf_full['prediction'] = (pdf_full['score_ensemble'] >= ens_threshold).astype(int)

out_cols = LABELS + ['score_lgbm', 'score_xgb', 'score_rf', 'score_ada', 'score_ensemble', 'rank', 'decile', 'prediction']
out = pdf_full[out_cols].sort_values('score_ensemble', ascending=False)
out.to_parquet('/home/jovyan/work/scores_unified.parquet', index=False)

print(f'\nScored {len(pdf_full):,} clients')
print(f'Decision threshold (ensemble, F1-optimal): {ens_threshold:.4f}')
print(f'Predicted positive (yes): {int(out["prediction"].sum()):,} clients ({out["prediction"].mean()*100:.2f}%)')
print(f'Predicted negative (no) : {int((1-out["prediction"]).sum()):,} clients')

del pdf_full, out, score_lgbm, score_xgb, score_rf, score_ada
gc.collect()

---
## 12. SHAP Explainability

In [ ]:
import shap

X_shap = X_val_fs.sample(5000, random_state=42)
explainer = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(X_shap)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_df = pd.DataFrame({
    'feature': X_shap.columns,
    'mean_abs_shap': np.abs(sv).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)
shap_df.to_csv(f'{REPORT_DIR}/25_shap.csv', index=False)

print('Top 20 features by |SHAP|:')
print(shap_df.head(20).to_string())

shap.summary_plot(sv, X_shap, max_display=20, show=False)
plt.title('SHAP Summary -- Unified Label')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/26_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

shap.summary_plot(sv, X_shap, plot_type='bar', max_display=20, show=False)
plt.title('SHAP Bar')
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/27_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final inventory
import glob
files = sorted(glob.glob(f'{REPORT_DIR}/*'))
print(f'=== {len(files)} report files ===')
for f in files:
    size = os.path.getsize(f)
    ftype = 'CSV' if f.endswith('.csv') else 'PNG' if f.endswith('.png') else 'OTHER'
    print(f'  [{ftype}] {os.path.basename(f):<50} {size:>8,} bytes')

In [ ]:
import optuna.visualization.matplotlib as ovm
import matplotlib.pyplot as plt

# Historique d'optimisation
fig, ax = plt.subplots(figsize=(10, 5))
ovm.plot_optimization_history(study_lgbm, ax=ax)
ax.set_title("Historique d'optimisation Optuna — LightGBM", fontsize=12, fontweight='bold')
ax.set_xlabel("Numéro d'essai")
ax.set_ylabel("Lift@40% obtenu")
plt.tight_layout()
plt.savefig('/home/jovyan/work/reports/optuna_history_lgbm.png', dpi=150, bbox_inches='tight')
plt.show()